# synthetic_ct_dataset -- phong_sh, albedo_0 test

Tests Phase 1 (render) + Phase 2 (decompose) for `sphere_phong_albedo_0` with the new features:
- `--light-mode directional/random_sh/circular`
- `--n-lights` (default 6)
- `--init-from-gt`
- `--log-gradients`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

from PIL import Image

# Repo root, found by walking up from the CWD. Notebooks live in notebooks/, but the
# code and the relative data paths below are relative to the repo root, so chdir there.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "idr").is_dir())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))




from idr.config import PHONG_MATERIAL_CONFIGS, DEFAULT_CFG, SHININESS_RANGE, NAMED_TRANSFORMS
from idr.paths import DATASET_ROOT, RESULTS_ROOT
from idr.pipelines.synthetic_generate import generate_dataset
from idr.pipelines.synthetic_run import run_study

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
print(f"dataset root: {DATASET_ROOT}")
print(f"results root: {RESULTS_ROOT}")

# -- experiment constants ------------------------------------------------------
N_LIGHTS    = 16
LIGHT_MODE  = "directional"   # "directional" | "random_sh" | "circular"
FULL_CIRCLE = False

def _suffix(mode=LIGHT_MODE, n=N_LIGHTS, full=FULL_CIRCLE):
    """Return the scene-name suffix matching _scene_suffix() in synthetic_ct_dataset."""
    if mode == "directional":
        tag = "full" if full else ""
        return f"dir{n}{tag}"
    elif mode == "random_sh":
        return f"rsh{n}"
    else:
        return f"circ{n}{'full' if full else ''}"

def scene_id(mesh="sphere", mat="albedo_0", mode=LIGHT_MODE, n=N_LIGHTS, full=FULL_CIRCLE, shader="phong"):
    shader_tag = f"_{shader}" if shader else ""
    return f"{mesh}{shader_tag}_{mat}_{_suffix(mode, n, full)}"

print(f"n_lights={N_LIGHTS}  light_mode={LIGHT_MODE}  scene_id={scene_id()}")

In [ ]:
# skip wandb logging
import unittest.mock as _m
_wand = _m.MagicMock()
_wand.Image = lambda x, **kw: x      # return array as-is so logging doesn't crash
_wand.init.return_value = _wand
sys.modules['wandb'] = _wand

## 1 -- Phase 1: render dataset

In [ ]:
# -- 1a: directional -----------------------------------------------------------
generate_dataset(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    light_mode="directional", n_lights=N_LIGHTS, full_circle=FULL_CIRCLE,
)
print("done: directional")

In [ ]:
# -- 1b: random_sh -------------------------------------------------------------
generate_dataset(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    light_mode="random_sh", n_lights=N_LIGHTS,
)
print("done: random_sh")

In [ ]:
# -- 1c: circular -------------------------------------------------------------
generate_dataset(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    light_mode="circular", n_lights=N_LIGHTS,
)
print("done: circular")

### Inspect rendered images

In [ ]:


def show_renders(scene_name, shader="phong_sh", title=None):
    scene_dir = DATASET_ROOT / scene_name / shader
    light_dirs = sorted(scene_dir.iterdir()) if scene_dir.exists() else []
    if not light_dirs:
        print(f"No renders found for {scene_name}/{shader}")
        return
    n = len(light_dirs)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    if n == 1: axes = [axes]
    fig.suptitle(title or scene_name, fontsize=11)
    for ax, ld in zip(axes, light_dirs):
        img_path = ld / "render.png"
        if img_path.exists():
            ax.imshow(Image.open(img_path))
        ax.set_title(ld.name, fontsize=7)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_renders(f"sphere_phong_albedo_0_dir{N_LIGHTS}",  title=f"directional, {N_LIGHTS} lights")
show_renders(f"sphere_phong_albedo_0_rsh{N_LIGHTS}",  title=f"random_sh,  {N_LIGHTS} lights")
show_renders(f"sphere_phong_albedo_0_circ{N_LIGHTS}", title=f"circular,   {N_LIGHTS} lights")

In [ ]:
# Show GT material maps (albedo, normal)
def show_gt(scene_name):
    gt_dir = DATASET_ROOT / scene_name / "gt"
    if not gt_dir.exists():
        print(f"No GT at {gt_dir}"); return
    files = sorted(gt_dir.glob("*.png"))
    n = len(files)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    if n == 1: axes = [axes]
    fig.suptitle(f"GT maps -- {scene_name}", fontsize=11)
    for ax, f in zip(axes, files):
        ax.imshow(Image.open(f))
        ax.set_title(f.stem, fontsize=8)
        ax.axis("off")
    plt.tight_layout(); plt.show()

show_gt(f"sphere_phong_albedo_0_dir{N_LIGHTS}")

## 2 -- Phase 2: decomposition

In [ ]:
CFG = dict(n_iter=100, lbfgs_max_iter=10, log_every=5)

In [ ]:
# # -- 2a: standard init ---------------------------------------------------------
# run_decomposition(
#     mesh_name="sphere", width=128, height=128,
#     shader="phong_sh", device=DEVICE,
#     mat_configs_filter={"albedo_0"},
#     cfg_overrides=CFG,
#     light_mode="directional", n_lights=N_LIGHTS, full_circle=FULL_CIRCLE,
#     init_from_gt=False,
# )
# print("done: standard init")

In [ ]:
# # -- 2b: GT init ---------------------------------------------------------------
# run_decomposition(
#     mesh_name="sphere", width=128, height=128,
#     shader="phong_sh", device=DEVICE,
#     mat_configs_filter={"albedo_0"},
#     cfg_overrides=CFG,
#     light_mode="directional", n_lights=N_LIGHTS, full_circle=FULL_CIRCLE,
#     init_from_gt=True,
# )
# print("done: GT init")

In [ ]:
# -- 2c: gradient logging ------------------------------------------------------
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    cfg_overrides={**CFG, "n_iter": 100},
    light_mode="directional", n_lights=N_LIGHTS, full_circle=FULL_CIRCLE,
    init_from_gt=False,
    log_gradients=True,
)
print("done: gradient logging")

In [ ]:
# -- 2d: GT init + gradient logging --------------------------------------------
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    cfg_overrides={**CFG, "n_iter":20},
    light_mode="random_sh", n_lights=N_LIGHTS, full_circle=FULL_CIRCLE,
    init_from_gt=True,
    log_gradients=True,
)
print("done: GT init + log gradients")

In [ ]:
# -- 2e: random_sh lighting ----------------------------------------------------
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    cfg_overrides=CFG,
    light_mode="random_sh", n_lights=N_LIGHTS,
    log_gradients=True,
)
print("done: random_sh")

In [ ]:
# -- 2f: log-shininess transform (only_shininess) ------------------------------
# Reparametrizes shininess in log space: raw param p -> shininess = exp(p).
# Gradients then scale with current shininess value rather than being tiny
# absolute steps. Standard init: exp(log(sqrt(s_min*s_max))) = sqrt(1*256) = 16.
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    cfg_overrides={**CFG, "n_iter": 100},
    light_mode="random_sh", n_lights=N_LIGHTS,
    log_gradients=True,
    transforms=NAMED_TRANSFORMS["only_shininess"],
    init_from_gt=True
)
print("done: log-shininess transform")

In [ ]:
# try ADAM

# run_decomposition(
#     mesh_name="sphere", width=128, height=128,
#     shader="phong_sh", device=DEVICE,
#     mat_configs_filter={"albedo_0"},
#     cfg_overrides=dict(optimizer = "ADAM", n_iter = 500, lr = 1e-4,),
#     light_mode="random_sh", n_lights=N_LIGHTS,
#     log_gradients=True,
# )
# print("done: random_sh")

## 3 -- Inspect results

In [ ]:
def results_dir(scene, shader_folder, tr="no_transforms"):
    return RESULTS_ROOT / tr / scene / shader_folder

def show_results(scene, shader_folder, tr="no_transforms"):
    d = results_dir(scene, shader_folder, tr)
    if not d.exists():
        print(f"Not found: {d}"); return

    scene_ds = DATASET_ROOT / scene

    # -- find base shader dir in dataset (strip _gtinit/_gradlog suffixes) ----
    shader_base = None
    parts = shader_folder.split("_")
    for i in range(len(parts), 0, -1):
        cand = "_".join(parts[:i])
        if (scene_ds / cand).is_dir():
            shader_base = cand; break

    # -- light keys -----------------------------------------------------------
    meta_path = scene_ds / "dataset_meta.json"
    if meta_path.exists():
        light_keys = json.loads(meta_path.read_text())["light_keys"]
    elif shader_base:
        light_keys = sorted(p.name for p in (scene_ds / shader_base).iterdir() if p.is_dir())
    else:
        light_keys = []
    N_L = max(len(light_keys), 1)

    SHIN_MAX = float(SHININESS_RANGE[1])   # 256

    # -- file helpers ----------------------------------------------------------
    def load_float(path):
        path = Path(path)
        npy = path.with_suffix(".npy")
        if npy.exists():  return np.load(npy).astype(np.float32)
        if path.exists(): return np.array(Image.open(path), dtype=np.float32) / 255.0
        return None

    def load_first(*paths):
        for p in paths:
            result = load_float(p)
            if result is not None:
                return result
        return None

    def load_param(base_dir, stem, pn):
        """Load in actual physical units: .npy preferred; PNG fallback denormalises shininess."""
        base_dir = Path(base_dir)
        npy = base_dir / f"{stem}.npy"
        png = base_dir / f"{stem}.png"
        if npy.exists():
            return np.load(npy).astype(np.float32)
        if png.exists():
            v = np.array(Image.open(png), dtype=np.float32) / 255.0
            return v * SHIN_MAX if pn == "shininess" else v
        return None

    def to_rgb(img):
        if img is None: return None
        if img.ndim == 2: return np.stack([img] * 3, -1)
        return img[..., :3]

    def param_for_display(data, pn):
        """Normalise actual-unit param to [0,1] for imshow."""
        if data is None: return None
        v = to_rgb(data)
        return (v / SHIN_MAX).clip(0, 1) if pn == "shininess" else v.clip(0, 1)

    # -- metrics ---------------------------------------------------------------
    metrics = json.loads((d / "metrics.json").read_text()) if (d / "metrics.json").exists() else {}
    albedo_scale = np.array(metrics.get("albedo_scale", [1.0, 1.0, 1.0]), dtype=np.float32)
    ks_scale = float(np.mean(albedo_scale))

    print(f"\n{'='*60}\n{scene} / {shader_folder}")
    for k in ("recon_rmse", "albedo_rmse", "final_loss"):
        if k in metrics:
            print(f"  {k}: {metrics[k]:.3e}")
    param_labels = sorted(k[:-9] for k in metrics if k.endswith("_est_mean"))
    for pl in param_labels:
        print()
        for suffix in ("_est_mean", "_gt", "_err_mean"):
            k = pl + suffix
            if k in metrics:
                print(f"  {k}: {metrics[k]:.3e}")

    # -- GT renders -----------------------------------------------------------
    gt_renders = [
        load_float(scene_ds / shader_base / lk / "render.png") if shader_base else None
        for lk in light_keys
    ]

    # -- reconstructions (prefer .npy, float unclamped) -----------------------
    recon_dir = d / "reconstructions"
    recon_list = [None] * N_L
    if recon_dir.exists():
        recon_files = {f.stem[len("recon_"):]: load_float(f)
                       for f in sorted(recon_dir.glob("recon_*.png"))
                       if not f.stem.startswith("recon_err")}
        keyed = [recon_files.get(lk) for lk in light_keys]
        if any(r is not None for r in keyed):
            recon_list = keyed
        else:
            vals = list(recon_files.values())
            recon_list = vals[:N_L] + [None] * (N_L - len(vals))

    # -- signed recon error (recon - GT, mean over RGB) -----------------------
    recon_err_list = [
        (to_rgb(rc) - to_rgb(gt)).mean(-1)
        if rc is not None and gt is not None else None
        for rc, gt in zip(recon_list, gt_renders)
    ]

    # -- params in actual units ------------------------------------------------
    spatial_params = [f.stem[:-4] for f in sorted(d.glob("*_est.png"))]
    gt_dir = scene_ds / "gt"
    est_actual = {pn: load_param(d,      f"{pn}_est", pn) for pn in spatial_params}
    gt_actual  = {pn: load_param(gt_dir,  pn,         pn) for pn in spatial_params}
    err_actual = {pn: load_param(d,      f"{pn}_err", pn) for pn in spatial_params}

    # -- env maps -------------------------------------------------------------
    gt_env_maps = [
        load_first(scene_ds / shader_base / lk / "sh_env_map.png",
                   scene_ds / shader_base / lk / "env_map.png")
        if shader_base else None
        for lk in light_keys
    ]
    est_env_maps = [
        load_first(d / f"sh_env_map_{lk}.png",
                   d / f"env_map_{lk}.png")
        for lk in light_keys
    ]
    has_env = any(e is not None for e in est_env_maps)

    # -- signed param errors ---------------------------------------------------
    def param_err(pn):
        ep, gp = est_actual.get(pn), gt_actual.get(pn)
        if ep is not None and gp is not None:
            if pn == "albedo":
                ep_corr = (to_rgb(ep) * albedo_scale).clip(0, 1)
                return (ep_corr - to_rgb(gp)).mean(-1)
            elif pn == "ks":
                ep_corr = to_rgb(ep) * ks_scale
                return (ep_corr - to_rgb(gp)).mean(-1)
            else:
                return (to_rgb(ep) - to_rgb(gp)).mean(-1)
        e = err_actual.get(pn)
        return to_rgb(e).mean(-1) if e is not None else None

    env_err = [
        (to_rgb(ge) - to_rgb(ee)).mean(-1)
        if ge is not None and ee is not None else None
        for ge, ee in zip(gt_env_maps, est_env_maps)
    ]

    # -- display arrays (normalised [0,1]) -------------------------------------
    gt_disp  = {pn: param_for_display(gt_actual.get(pn), pn)  for pn in spatial_params}
    est_disp = {}
    for pn in spatial_params:
        ep = est_actual.get(pn)
        if pn == "albedo" and ep is not None:
            ep = (to_rgb(ep) * albedo_scale).clip(0, 1)
        elif pn == "ks" and ep is not None:
            ep = (to_rgb(ep) * ks_scale).clip(0, 1)
        est_disp[pn] = param_for_display(ep, pn)

    # -- column layout ---------------------------------------------------------
    param_col_names = spatial_params + (light_keys if has_env else [])
    N_cols = max(N_L, len(param_col_names), 1)

    def param_row(items_spatial, items_env):
        items = items_spatial + (items_env if has_env else [])
        return items + [None] * (N_cols - len(items))

    rows_spec = [
        ("GT images",  gt_renders     + [None] * (N_cols - N_L), light_keys, False),
        ("recons",     recon_list     + [None] * (N_cols - N_L), light_keys, False),
        ("recon err",  recon_err_list + [None] * (N_cols - N_L), light_keys, True),
        ("params GT",  param_row([gt_disp.get(p)   for p in spatial_params], gt_env_maps),
                       param_col_names, False),
        ("params est", param_row([est_disp.get(p)  for p in spatial_params], est_env_maps),
                       param_col_names, False),
        ("param err",  param_row([param_err(p)      for p in spatial_params], env_err),
                       param_col_names, True),
    ]

    n_rows = len(rows_spec)
    fig, axes = plt.subplots(n_rows, N_cols,
                             figsize=(2.8 * N_cols, 2.6 * n_rows), squeeze=False)
    fig.suptitle(f"{scene} / {shader_folder}", fontsize=11)

    for ri, (rlabel, items, col_lbls, is_err) in enumerate(rows_spec):
        for ci in range(N_cols):
            ax  = axes[ri, ci]
            img = items[ci] if ci < len(items) else None

            if ri == 0 and ci < len(col_lbls):
                ax.set_title(col_lbls[ci], fontsize=7)
            elif rlabel in ("params GT", "params est", "param err") and ci < len(col_lbls):
                ax.set_title(col_lbls[ci], fontsize=7)

            if img is None:
                ax.axis("off"); continue

            if is_err:
                vmax = float(np.nanmax(np.abs(img))) or 1e-6
                im = ax.imshow(img, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
                plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
            else:
                ax.imshow(to_rgb(img).clip(0, 1))

            ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
            for spine in ax.spines.values():
                spine.set_visible(False)

    plt.tight_layout()
    plt.subplots_adjust(left=0.1)
    for ri, (rlabel, *_) in enumerate(rows_spec):
        pos = axes[ri, 0].get_position()
        fig.text(0.005, (pos.y0 + pos.y1) / 2, rlabel,
                 fontsize=8, fontweight="bold", ha="left", va="center")
    plt.show()

# show_results(scene_id(), "phong_sh_gradlog")
show_results(scene_id(mode="random_sh"), "phong_sh_gradlog")
# random_sh results
# show_results(scene_id(mode="random_sh"), "phong_sh")
# show_results(scene_id(), "phong_sh_gradlog")
# show_results(scene_id(mode="random_sh"), "phong_sh_gradlog")
# GT init + gradlog
# show_results(scene_id(), "phong_sh_gtinit_gradlog")

## 4 -- Gradient flow inspection

In [ ]:
N_LIGHTS

In [ ]:
# grad_dir = results_dir(scene_id(), "phong_sh_gradlog") / "gradient_flow"
# grad_dir = results_dir(scene_id(mode="random_sh"), "phong_sh_gradlog") / "gradient_flow"
# grad_dir = results_dir(scene_id(mode="random_sh"), "phong_sh_gtinit_gradlog") / "gradient_flow"
# grad_dir = results_dir(scene_id(), "phong_sh_gtinit_gradlog") / "gradient_flow"
# grad_dir = results_dir(scene_id(mode="random_sh"), "phong_sh_gradlog", tr="only_shininess_transforms") / "gradient_flow"
grad_dir = results_dir(scene_id(mode="random_sh"), "phong_sh_gtinit_gradlog", tr="only_shininess_transforms") / "gradient_flow"
npz_files = sorted(grad_dir.glob("step_*.npz")) if grad_dir.exists() else []
print(f"Gradient snapshots: {len(npz_files)}")
if npz_files:
    sample = np.load(npz_files[0])
    print("Keys:", list(sample.keys()))

In [ ]:
def plot_grad_flow(grad_dir, param="albedo"):
    npz_files = sorted(Path(grad_dir).glob("step_*.npz"))
    if not npz_files:
        print("No gradient snapshots found.")
        return

    albedo_scale = np.ones(3, dtype=np.float32)
    metrics_path = Path(grad_dir).parent / "metrics.json"
    if metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        if "albedo_scale" in m:
            albedo_scale = np.array(m["albedo_scale"], dtype=np.float32)
    ks_scale = float(np.mean(albedo_scale))

    steps, grad_norms, update_norms, gt_errors = [], [], [], []
    losses = []
    for f in npz_files:
        d = np.load(f)
        steps.append(int(f.stem.split("_")[1]))
        gk = f"{param}_grad"
        uk = f"{param}_update"
        ek = f"{param}_gt_error"
        if gk in d: grad_norms.append(float(np.linalg.norm(d[gk])))
        if uk in d: update_norms.append(float(np.linalg.norm(d[uk])))
        if ek in d:
            err = d[ek]
            if param == "albedo" and f"{param}_value" in d:
                alb_gt = d["albedo_value"] - err
                err = d["albedo_value"] * albedo_scale - alb_gt
            elif param == "albedo":
                err = err * albedo_scale
            elif param == "ks" and f"{param}_value" in d:
                ks_GT_map = d[f"{param}_value"] - err
                err = d[f"{param}_value"] * ks_scale - ks_GT_map
            gt_errors.append(float(np.abs(err).mean()))
        if "loss_total" in d: losses.append(float(d["loss_total"]))

    scaled_note = "  (scaled)" if param in ("albedo", "ks") else ""
    fig, axes = plt.subplots(1, 4, figsize=(16, 3))
    fig.suptitle(f"Gradient flow -- {param}", fontsize=11)
    for ax, (vals, label) in zip(axes, [
        (losses, "loss_total"),
        (grad_norms, f"|grad({param})|"),
        (update_norms, f"|update({param})|"),
        (gt_errors, f"mean |{param} - GT|{scaled_note}"),
    ]):
        if vals:
            ax.plot(steps[:len(vals)], vals, marker=".")
        ax.set_title(label, fontsize=9)
        ax.set_xlabel("iter")
        ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_grad_flow(grad_dir, param="albedo")
plot_grad_flow(grad_dir, param="ks")
plot_grad_flow(grad_dir, param="sh")

In [ ]:
def plot_top_error_pixel_details(grad_dir, scene_name, shader, n_top=5, rank_by="albedo"):
    from idr.paths import DATASET_ROOT

    npz_files = sorted(Path(grad_dir).glob("step_*.npz"))
    if not npz_files:
        print("No gradient snapshots."); return
    steps = [int(f.stem.split("_")[1]) for f in npz_files]

    # -- target images + foreground mask --------------------------------------
    meta_path = DATASET_ROOT / scene_name / "dataset_meta.json"
    light_keys = json.loads(meta_path.read_text())["light_keys"] if meta_path.exists() else []
    target_imgs = []
    for lk in light_keys:
        img_path = DATASET_ROOT / scene_name / shader / lk / "render.png"
        npy_path  = img_path.with_suffix(".npy")
        if npy_path.exists():
            target_imgs.append(np.load(npy_path).astype(np.float32))
        elif img_path.exists():
            target_imgs.append(np.array(Image.open(img_path), dtype=np.float32) / 255.0)
    if not target_imgs:
        print(f"No target images for {scene_name}/{shader}"); return

    H, W   = target_imgs[0].shape[:2]
    N_L    = len(target_imgs)
    mask2d = np.zeros((H, W), dtype=bool)
    for img in target_imgs:
        mask2d |= (img.max(-1) > 1e-6)

    # -- albedo/ks scale (needed for ranking and display) ----------------------
    albedo_scale = np.ones(3, dtype=np.float32)
    metrics_path = Path(grad_dir).parent / "metrics.json"
    if metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        if "albedo_scale" in m:
            albedo_scale = np.array(m["albedo_scale"], dtype=np.float32)
            print(f"albedo_scale: {np.round(albedo_scale, 4).tolist()}")
    ks_scale = float(np.mean(albedo_scale))

    # -- rank foreground pixels by final-step error ----------------------------
    final = np.load(npz_files[-1])
    if rank_by == "recon":
        recon_err_final = np.zeros((H, W), dtype=np.float32)
        n_valid = 0
        for ki in range(N_L):
            pref = f"shade_k{ki:02d}_"
            if f"{pref}diff" in final and f"{pref}spec" in final:
                recon = final[f"{pref}diff"] + final[f"{pref}spec"]
                recon_err_final += np.abs(recon - target_imgs[ki]).mean(-1)
                n_valid += 1
        err_mag = (recon_err_final / max(n_valid, 1)) * mask2d
    else:
        err_arr = final[f"{rank_by}_gt_error"]
        if rank_by == "albedo" and "albedo_value" in final:
            alb_gt = final["albedo_value"] - err_arr
            err_arr = final["albedo_value"] * albedo_scale - alb_gt
        elif rank_by == "albedo":
            err_arr = err_arr * albedo_scale
        elif rank_by == "ks" and "ks_value" in final:
            ks_GT_map = final["ks_value"] - err_arr
            err_arr   = final["ks_value"] * ks_scale - ks_GT_map
        err_mag = np.abs(err_arr).mean(-1) * mask2d

    flat_idx = np.argsort(err_mag.ravel())[::-1][:n_top]
    pix_rc   = [(i // W, i % W) for i in flat_idx]

    # -- collect per-step data -------------------------------------------------
    spatial_params = [p for p in ("albedo", "shininess", "ks", "metallic", "roughness")
                      if f"{p}_update" in final]
    quantities     = ["value", "update", "grad", "gt_error"]

    pdata = {p: {q: [] for q in quantities} for p in spatial_params}
    recon_err = []   # (n_steps, n_top, N_L)

    for f in npz_files:
        d = np.load(f)
        for p in spatial_params:
            for q in quantities:
                key = f"{p}_{q}"
                if key in d:
                    arr = d[key]
                    pdata[p][q].append(np.array([arr[r, c] for r, c in pix_rc]))

        step_err = np.zeros((n_top, N_L))
        for ki in range(N_L):
            pref = f"shade_k{ki:02d}_"
            if f"{pref}diff" in d and f"{pref}spec" in d:
                recon = d[f"{pref}diff"] + d[f"{pref}spec"]
                per_px = np.abs(recon - target_imgs[ki]).mean(-1)
                for pi, (r, c) in enumerate(pix_rc):
                    step_err[pi, ki] = per_px[r, c]
        recon_err.append(step_err)

    recon_err = np.stack(recon_err)                                     # (n_steps, n_top, N_L)
    pdata     = {p: {q: np.stack(v) for q, v in qs.items() if v}
                 for p, qs in pdata.items()}

    # -- scale corrections on pdata (albedo and ks share the SH scale) --------
    if "albedo" in pdata and "value" in pdata["albedo"] and "gt_error" in pdata["albedo"]:
        gt_albedo_raw = pdata["albedo"]["value"][0] - pdata["albedo"]["gt_error"][0]
        pdata["albedo"]["value"]    = pdata["albedo"]["value"]    * albedo_scale
        pdata["albedo"]["gt_error"] = pdata["albedo"]["value"]    - gt_albedo_raw

    if "ks" in pdata and "value" in pdata["ks"] and "gt_error" in pdata["ks"]:
        gt_ks_raw = pdata["ks"]["value"][0] - pdata["ks"]["gt_error"][0]   # (n_top, 1)
        pdata["ks"]["value"]    = pdata["ks"]["value"]    * ks_scale
        pdata["ks"]["gt_error"] = pdata["ks"]["value"]    - gt_ks_raw

    gt_vals = {}
    for p in spatial_params:
        if "value" in pdata[p] and "gt_error" in pdata[p]:
            gt_vals[p] = pdata[p]["value"][0] - pdata[p]["gt_error"][0]  # (n_top, C)

    # -- location map ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(err_mag, cmap="hot")
    for i, (r, c) in enumerate(pix_rc):
        ax.scatter(c, r, s=120, marker="*", color="cyan", zorder=5)
        ax.text(c + 1, r - 1, str(i + 1), color="cyan", fontsize=8, fontweight="bold")
    if rank_by == "recon":
        rank_label = "mean recon err"
    elif rank_by == "albedo":
        rank_label = "|albedo-GT| (scaled)"
    elif rank_by == "ks":
        rank_label = "|ks-GT| (scaled)"
    else:
        rank_label = f"|{rank_by}-GT|"
    ax.set_title(f"Top-{n_top} foreground pixels by {rank_label}", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout(); plt.show()

    # -- per-pixel figures -----------------------------------------------------
    ch_colors    = ["#e41a1c", "#4daf4a", "#377eb8"]
    light_colors = plt.cm.tab20.colors
    ch_labels    = {"albedo": ["R", "G", "B"], "shininess": ["shin"], "ks": ["ks"],
                    "metallic": ["met"], "roughness": ["rough"]}
    qty_labels   = {"value": "value (albedo: scale-corrected)", "update": "Delta/step",
                    "grad": "gradient", "gt_error": "error vs GT"}

    # row 0: 2 cols (all lights | mean), rows 1+: params x-- len(quantities) cols
    n_cols = max(len(quantities), 2)
    n_rows = 1 + len(spatial_params)

    for pix_i, (r, c) in enumerate(pix_rc):
        fig, axes = plt.subplots(n_rows, n_cols,
                                 figsize=(3.5 * n_cols, 2.5 * n_rows), squeeze=False)
        err_label = (f"recon err = {err_mag[r, c]:.4f}" if rank_by == "recon"
                     else f"|{rank_by}-GT| = {err_mag[r, c]:.4f}")
        fig.suptitle(f"Pixel #{pix_i+1}  ({r}, {c})   {err_label}", fontsize=11)

        # Row 0 col 0: all lights as individual lines
        ax0 = axes[0, 0]
        for ki in range(N_L):
            ax0.plot(steps, recon_err[:, pix_i, ki],
                     color=light_colors[ki % len(light_colors)],
                     lw=1, alpha=0.7, label=f"L{ki}")
        ax0.set_title("recon |err| -- all lights", fontsize=8)
        ax0.set_xlabel("iter", fontsize=7)
        ax0.axhline(0, color="k", lw=0.5, ls="--", alpha=0.4)
        ax0.grid(True, alpha=0.3)
        if N_L <= 10:
            ax0.legend(fontsize=5, ncol=2, loc="upper right")

        # Row 0 col 1: mean across lights
        ax1 = axes[0, 1]
        ax1.plot(steps, recon_err[:, pix_i, :].mean(-1), color="purple", lw=1.5)
        ax1.set_title("recon |err| mean", fontsize=8)
        ax1.set_xlabel("iter", fontsize=7)
        ax1.axhline(0, color="k", lw=0.5, ls="--", alpha=0.4)
        ax1.grid(True, alpha=0.3)

        for col_i in range(2, n_cols):
            axes[0, col_i].axis("off")

        # Rows 1+: spatial params x-- quantities
        for row_off, p in enumerate(spatial_params):
            row_i = row_off + 1
            for col_i, q in enumerate(quantities):
                ax = axes[row_i, col_i]
                if q in pdata[p]:
                    vals = pdata[p][q][:, pix_i, :]   # (n_steps, C)
                    for ch in range(vals.shape[-1]):
                        lbl = ch_labels.get(p, [str(ch)])[ch] \
                              if ch < len(ch_labels.get(p, [])) else str(ch)
                        ax.plot(steps, vals[:, ch], marker=".",
                                color=ch_colors[ch % 3], label=lbl, lw=1.5)
                    if q == "value" and p in gt_vals:
                        for ch, gv in enumerate(gt_vals[p][pix_i]):
                            ax.axhline(gv, color=ch_colors[ch % 3],
                                       lw=1, ls="--", alpha=0.6)
                ax.axhline(0, color="k", lw=0.5, ls="--", alpha=0.4)
                ax.set_title(f"{p} -- {qty_labels.get(q, q)}", fontsize=8)
                ax.set_xlabel("iter", fontsize=7)
                ax.legend(fontsize=6, loc="best")
                ax.grid(True, alpha=0.3)
            for col_i in range(len(quantities), n_cols):
                axes[row_i, col_i].axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
# plot_top_error_pixel_details(grad_dir, scene_id(mode="random_sh"), "phong_sh", n_top=1, rank_by="recon")
# plot_top_error_pixel_details(grad_dir, scene_id(mode="random_sh"), "phong_sh", n_top=1, rank_by="albedo")
plot_top_error_pixel_details(grad_dir, scene_id(mode="random_sh"), "phong_sh", n_top=1, rank_by="ks")
# plot_top_error_pixel_details(grad_dir, scene_id(mode="random_sh"), "phong_sh", n_top=1, rank_by="shininess")

In [ ]:
# Visualise albedo error map at first and last step
def show_grad_maps(grad_dir, param="albedo", key="gt_error"):
    npz_files = sorted(Path(grad_dir).glob("step_*.npz"))
    if not npz_files:
        print("No snapshots."); return

    albedo_scale = np.ones(3, dtype=np.float32)
    metrics_path = Path(grad_dir).parent / "metrics.json"
    if metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        if "albedo_scale" in m:
            albedo_scale = np.array(m["albedo_scale"], dtype=np.float32)
    ks_scale = float(np.mean(albedo_scale))

    picks = [npz_files[0], npz_files[-1]]
    labels = ["step 0 (init)", f"step {len(npz_files)-1} (final)"]
    scale_note = "  (scaled)" if param in ("albedo", "ks") and key == "gt_error" else ""
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    fig.suptitle(f"{param}_{key}{scale_note}", fontsize=11)
    for ax, f, lbl in zip(axes, picks, labels):
        d = np.load(f)
        k = f"{param}_{key}"
        if k not in d:
            ax.set_title(f"{lbl}\n(key missing)"); ax.axis("off"); continue
        arr = d[k]
        if param == "albedo" and key == "gt_error":
            alb_val = d.get(f"{param}_value")
            if alb_val is not None:
                alb_gt = alb_val - arr
                arr = alb_val * albedo_scale - alb_gt
            else:
                arr = arr * albedo_scale
        elif param == "ks" and key == "gt_error":
            ks_val = d.get(f"{param}_value")
            if ks_val is not None:
                ks_GT_map = ks_val - arr
                arr = ks_val * ks_scale - ks_GT_map
        if arr.ndim == 3 and arr.shape[-1] >= 3:
            ax.imshow(np.abs(arr).mean(-1), cmap="hot")
        elif arr.ndim == 3:
            ax.imshow(np.abs(arr[..., 0]), cmap="hot")
        else:
            ax.imshow(np.abs(arr), cmap="hot")
        ax.set_title(lbl, fontsize=9); ax.axis("off")
        plt.colorbar(ax.images[0], ax=ax, fraction=0.04)
    plt.tight_layout(); plt.show()

show_grad_maps(grad_dir, "albedo", "gt_error")
show_grad_maps(grad_dir, "albedo", "grad")

In [ ]:
# Shade components at step 0 -- e.g. diffuse, specular
def show_shade_components(grad_dir, step=0, light_idx=0):
    f = sorted(Path(grad_dir).glob("step_*.npz"))[step]
    d = np.load(f)
    comp_keys = [k for k in d.keys() if k.startswith(f"shade_k{light_idx:02d}_")]
    if not comp_keys:
        print(f"No shade components for light k{light_idx:02d} in {f.name}"); return
    n = len(comp_keys)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))
    if n == 1: axes = [axes]
    fig.suptitle(f"{f.stem} -- shade components (light {light_idx})", fontsize=10)
    for ax, k in zip(axes, comp_keys):
        arr = d[k]
        if arr.ndim == 3 and arr.shape[-1] == 3:
            ax.imshow(arr.clip(0, 1))
        else:
            ax.imshow(arr.squeeze(), cmap="viridis")
        ax.set_title(k.split("_", 2)[-1], fontsize=7)
        ax.axis("off")
    plt.tight_layout(); plt.show()

show_shade_components(grad_dir, step=0, light_idx=0)

## 5 -- GT init vs standard: loss convergence comparison

In [ ]:
def load_metrics(scene, shader_folder, tr="no_transforms"):
    p = RESULTS_ROOT / tr / scene / shader_folder / "metrics.json"
    return json.loads(p.read_text()) if p.exists() else {}

m_std  = load_metrics(scene_id(),              "phong_sh_gradlog")
m_gt   = load_metrics(scene_id(),              "phong_sh_gtinit")
m_rsh  = load_metrics(scene_id(mode="random_sh"), "phong_sh_gradlog")

rows = [
    (f"std init / {scene_id()}",         m_std),
    (f"GT  init / {scene_id()}",         m_gt),
    (f"std init / {scene_id(mode='random_sh')}", m_rsh),
]
print(f"{'Run':<55} {'albedo_rmse':>12} {'final_loss':>12}")
print("-" * 82)
for name, m in rows:
    ar = m.get("albedo_rmse", float("nan"))
    fl = m.get("final_loss",  float("nan"))
    print(f"{name:<55} {ar:>12.5f} {fl:>12.5f}")

## 6 Cook-Torrance shader (ct_sh)

In [ ]:
# -- 6a: Phase 1 -- render ct_sh dataset ---------------------------------------
generate_dataset(
    mesh_name="sphere", width=128, height=128,
    shader="ct_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    # mat_configs_filter={"roughness_0"},
    light_mode="random_sh", n_lights=N_LIGHTS,
)
print("done: ct_sh renders")

In [ ]:
# 6b: Phase 2 - standard init + gradient logging 
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="ct_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    cfg_overrides={**CFG, "n_iter": 100},
    light_mode="random_sh", n_lights=N_LIGHTS,
    log_gradients=True,    
)
print("done: ct_sh standard init")

In [ ]:
# 6c: Phase 2 - GT init + gradient logging 
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="ct_sh", device=DEVICE,
    mat_configs_filter={"albedo_0"},
    cfg_overrides={**CFG, "n_iter": 200},
    light_mode="random_sh", n_lights=N_LIGHTS,    
    log_gradients=True,
)
print("done: ct_sh GT init")

In [ ]:
# -- 6d: inspect results -------------------------------------------------------
# ct_grad_dir = results_dir(scene_id(mode="random_sh", shader=None), "ct_sh_gradlog") / "gradient_flow"
ct_grad_dir = results_dir(scene_id(mode="random_sh", shader=None), "ct_sh_gtinit_gradlog") / "gradient_flow"
ct_npz = sorted(ct_grad_dir.glob("step_*.npz")) if ct_grad_dir.exists() else []
print(f"ct_sh snapshots: {len(ct_npz)}")
if ct_npz:
    print("Keys:", [k for k in np.load(ct_npz[0]).keys() if not k.startswith("shade")])

In [ ]:
# show_results(scene_id(mode="random_sh", shader=None), "ct_sh_gradlog")
show_results(scene_id(mode="random_sh", shader=None), "ct_sh_gtinit_gradlog")

In [ ]:
for p in ("albedo", "metallic", "roughness", "sh"):
    plot_grad_flow(ct_grad_dir, param=p)

In [ ]:
# plot_top_error_pixel_details(ct_grad_dir, scene_id(mode="random_sh", shader=None), "ct_sh", n_top=1, rank_by="albedo")
plot_top_error_pixel_details(ct_grad_dir, scene_id(mode="random_sh", shader=None), "ct_sh", n_top=1, rank_by="roughness")
# plot_top_error_pixel_details(ct_grad_dir, scene_id(mode="random_sh", shader=None), "ct_sh", n_top=1, rank_by="metallic")

## 7 -- Random patch textures

Spherical UV coordinates are derived from the surface normal:
- `u = 0.5 + atan2(nz, nx) / (2*pi)` (longitude)
- `v = 0.5 - asin(ny) / pi` (latitude)

An `n_tiles x n_tiles` grid is overlaid; each cell gets an independent random value
sampled uniformly from `[val_low, val_high]`.  This produces many small, randomly
varying patches -- much richer than a two-value checkerboard.

**Available texture configs** (`n_tiles=16`):
- CT `all_texture`: albedo in [0.1, 0.9]^3, metallic in [0.0, 1.0], roughness in [0.1, 0.9]
- Phong `all_texture`: albedo in [0.1, 0.9]^3, shininess in [4, 63], ks in [0.1, 0.9]

Old two-value configs (`albedo_checker`, `shininess_checker`, `ks_checker`) are still available.

In [ ]:
# -- 7a: Phase 1 -- render texture scenes -------------------------------------
generate_dataset(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"all_texture"},
    light_mode="random_sh", n_lights=N_LIGHTS,
)
# CT all_texture
generate_dataset(
    mesh_name="sphere", width=128, height=128,
    # mesh_name="sphere", width=256, height=256,
    shader="ct_sh", device=DEVICE,
    mat_configs_filter={"all_texture"},
    light_mode="random_sh", n_lights=N_LIGHTS,
)
print("done: Phase 1 texture renders")

In [ ]:
# -- 7b: Phase 2 -- decompose texture albedo (Phong) --------------------------
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="phong_sh", device=DEVICE,
    mat_configs_filter={"all_texture"},
    cfg_overrides={**CFG, "n_iter": 100},
    light_mode="random_sh", n_lights=N_LIGHTS,
    log_gradients=True,
)
print("done: texture decomp")

In [ ]:
# -- 7c: inspect results ------------------------------------------------------
show_results(scene_id(mode="random_sh", shader="phong", mat="all_texture"), "phong_sh_gradlog")

In [ ]:
tex_grad_dir = results_dir(
    scene_id(mode="random_sh", shader="phong", mat="all_texture"),
    "phong_sh_gradlog",
) / "gradient_flow"
for p in ("albedo", "sh", "shininess", "ks"):
    plot_grad_flow(tex_grad_dir, param=p)

In [ ]:
plot_top_error_pixel_details(
    tex_grad_dir,
    scene_id(mode="random_sh", shader="phong", mat="all_texture"),
    "phong_sh", n_top=1, rank_by="albedo",
)

In [ ]:
# -- 7d: Phase 2 -- decompose CT texture --------------------------------------
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    # mesh_name="sphere", width=256, height=256,
    shader="ct_sh", device=DEVICE,
    mat_configs_filter={"all_texture"},
    cfg_overrides={**CFG, "n_iter": 100},
    light_mode="random_sh", n_lights=N_LIGHTS,
    log_gradients=True,
)
print("done: CT texture decomp")

In [ ]:
# -- 7e: inspect results ------------------------------------------------------
show_results(scene_id(mode="random_sh", shader=None, mat="all_texture"), "ct_sh_gradlog")

In [ ]:
ct_tex_grad_dir = results_dir(
    scene_id(mode="random_sh", shader=None, mat="all_texture"),
    "ct_sh_gradlog",
) / "gradient_flow"
for p in ("albedo", "metallic", "roughness", "sh"):
    plot_grad_flow(ct_tex_grad_dir, param=p)

In [ ]:
plot_top_error_pixel_details(
    ct_tex_grad_dir,
    scene_id(mode="random_sh", shader=None, mat="all_texture"),
    "ct_sh", n_top=1, rank_by="albedo",
)

## 8 -- TV regularization study (CT, all_texture)

Total-variation regularization penalizes spatial gradients of albedo, metallic,
and roughness, encouraging piecewise-smooth reconstructions:

    L_tv = lambda_tv * (TV(albedo) + TV(metallic) + TV(roughness))

where  sums L1 norms of horizontal and vertical finite differences.
Below we compare lambda_tv = 0 (baseline, cell 7d), 1e-3, and 1e-2.

In [ ]:
# -- 8a: Phase 2 -- CT texture + TV lambda=1e-3 -------------------------
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="ct_sh", device=DEVICE,
    mat_configs_filter={"all_texture"},
    cfg_overrides={**CFG, "n_iter": 100, "lambda_tv": 1e-3},
    light_mode="random_sh", n_lights=N_LIGHTS,
    log_gradients=True,
)
print("done: CT texture + TV 1e-3")

In [ ]:
# -- 8b: Phase 2 -- CT texture + TV lambda=1e-2 -------------------------
run_decomposition(
    mesh_name="sphere", width=128, height=128,
    shader="ct_sh", device=DEVICE,
    mat_configs_filter={"all_texture"},
    cfg_overrides={**CFG, "n_iter": 100, "lambda_tv": 1e-2},
    light_mode="random_sh", n_lights=N_LIGHTS,
    log_gradients=True,
)
print("done: CT texture + TV 1e-2")

In [ ]:
# -- 8c: inspect results (all three lambda_tv values) --------------------
_sid = scene_id(mode="random_sh", shader=None, mat="all_texture")
for label in ("ct_sh_lt=0.001_gradlog", "ct_sh_lt=0.01_gradlog"):
    print(f"--- {label} ---")
    show_results(_sid, label)

In [ ]:
# -- 8d: gradient flow -- TV 1e-3 ----------------------------------------
_sid = scene_id(mode="random_sh", shader=None, mat="all_texture")
for lt_label in ("ct_sh_lt=0.001_gradlog", "ct_sh_lt=0.01_gradlog"):
    gd = results_dir(_sid, lt_label) / "gradient_flow"
    print(f"=== {lt_label} ===")
    for p in ("albedo", "metallic", "roughness", "sh"):
        plot_grad_flow(gd, param=p)

In [ ]:
# -- 8e: top-error pixels -- TV 1e-3 vs 1e-2 ----------------------------
_sid = scene_id(mode="random_sh", shader=None, mat="all_texture")
for lt_label in ("ct_sh_lt=0.001_gradlog", "ct_sh_lt=0.01_gradlog"):
    gd = results_dir(_sid, lt_label) / "gradient_flow"
    print(f"=== {lt_label} ===")
    plot_top_error_pixel_details(gd, _sid, "ct_sh", n_top=1, rank_by="albedo")